#PydanticOutputParser——让模型输出"带schema的JSON“

#Pydantic是“运行时数据校验库”：你定义一个 class 声明"有哪些字段、什么类型"，它自动帮你校验值合不合法。

In [3]:
from ast import parse

from babel.dates import format_interval
from pydantic import BaseModel,Field

class Resume(BaseModel):
    name: str  #字段名+类型注解
    years:int
    skills:list[str] = Field(description="技能列表")  #给字段加描述，后面会变成给模型看的字段说明

r = Resume(name="张三",years=2,skills=["python","rag"])

print(r.name)

print(r.model_dump())

张三
{'name': '张三', 'years': 2, 'skills': ['python', 'rag']}


#PydanticOutputParser接模型

In [6]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from dotenv import load_dotenv
import os

load_dotenv()

#1.定义输出结构
class JobSummary(BaseModel):
    position:str
    required_skills:list[str]

#2.创建解析器(自动生成"输出格式说明")
parser = PydanticOutputParser(pydantic_object=JobSummary)

#3.模板里把格式说明填进去
prompt = ChatPromptTemplate.from_messages([
    ("system","根据用户输入提取信息，严格按下面的格式输出:\n{format_instructions}"),
    ("user","{text}")
]).partial(format_instructions=parser.get_format_instructions()) #解析器自动生成“输出格式说明书”告诉模型“JSON里要有position、required_skills这些字段” .partia()给模板**预填部分变量**（format_instructions 固定不变），这样 invoke 时只需要传 `text`

#4.调用并解析
llm = ChatOpenAI(
    model="deepseek-v4-flash",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    temperature=0.3,
    max_tokens=300
)

raw = llm.invoke(prompt.invoke({"text":"招聘Python后端工程师,要求熟悉RAG和FastAPI"}))

parsed = parser.invoke(raw)

print(parsed.position)  #直接取字段

print(parsed.required_skills)

Python后端工程师
['RAG', 'FastAPI']
